# Lesson 15 Lab — Torch-Pruning DepGraph: A Structured-Pruning Compatibility Lab

**Puzzle:** Can a dependency graph identify every tensor coupled to one channel deletion on this environment?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

DepGraph turns a local root operation into a pruning group. That is precisely the bookkeeping manual structural pruning tends to miss. A credible lab must distinguish the graph concept, a manual CUDA control, and whether the optional Torch-Pruning package executed successfully on the recorded stack.


## 0. Predict before running

1. Predict which modules join a group rooted at the first convolution.
2. Predict the result when the optional package is absent.
3. State what must be saved after module shapes are mutated.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

A residual mini-network, one channel index set, a manually synchronized narrow copy, an import/version probe, and—when available—a real `DependencyGraph` group are the concrete objects.

- DepGraph groups coupled pruning operations from a root decision.
- Example inputs and enabled autograd define the traced dependency path.
- Package compatibility evidence is distinct from manual structural correctness.


## 2. Derive the mechanism

Torch-Pruning traces an example forward with autograd enabled, then maps a root pruning function through module and tensor dependencies. Group validation prevents deleting an entire dimension. The package mutates module structure, so saving a plain dense-definition state_dict is insufficient unless architecture metadata is reconstructed. The manual control proves expected shape propagation independently of package availability.

### Mechanism at a glance

```mermaid
flowchart LR
  M["model + example inputs"] --> D["DepGraph trace"]
  R["root prune request"] --> G["dependency group"]
  D --> G
  G --> C{"group constraints pass?"}
  C -->|"no"| X["reject or reduce indices"]
  C -->|"yes"| P["execute group pruning"]
  P --> V["forward + shape + quality checks"]
```

### Walk it step by step

1. **Trace with representative inputs.** Dependency discovery must see the operators, merges, and shapes used by the intended execution path.
2. **Request one root pruning action.** Choose a layer, pruning function, and concrete index set rather than editing tensors directly.
3. **Inspect the generated group.** Review every coupled operation and reject a group that violates minimum channels, grouping, or model interfaces.
4. **Execute and validate the mutation.** Run forward, parameter, shape, export, and quality checks before treating the DepGraph result as usable.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 15
LESSON_TITLE = 'Torch-Pruning DepGraph: A Structured-Pruning Compatibility Lab'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260823
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | manual synchronized pruning ledger for a residual mini-network |
| Candidate | Torch-Pruning dependency group and mutation when the package is available |
| Held constant | model, example input, root module, channel indices, eval mode, and GPU |
| Measurements | package availability/version, group validity/size, output shape, parameters, and caught exception |
| Evidence | `compatibility-probe` |

**Experiment:** Build a real DepGraph group when available and always execute a manual CUDA structural-control path.


## 5. Read the experiment code

The notebook first runs the manual control so the lesson remains informative on a minimal PyTorch installation. It then probes `torch_pruning`, builds the graph without `no_grad`, requests a pruning group, validates it, and records group detail instead of converting import failure into a successful backend claim.

Do not execute until the code implements the frozen table above.


In [2]:
class MiniResidual(nn.Module):
    def __init__(self,width=12): super().__init__(); self.a=nn.Conv2d(4,width,1,bias=False); self.b=nn.Conv2d(4,width,3,padding=1,bias=False); self.out=nn.Conv2d(width,6,1,bias=False)
    def forward(self,x): return self.out(F.relu(self.a(x)+self.b(x)))
model=MiniResidual().to(DEVICE).eval(); example=torch.randn(1,4,12,12,device=DEVICE); idxs=[1,3,5,7]
keep=torch.tensor([i for i in range(12) if i not in idxs],device=DEVICE); manual=MiniResidual(8).to(DEVICE).eval()
with torch.no_grad(): manual.a.weight.copy_(model.a.weight[keep]); manual.b.weight.copy_(model.b.weight[keep]); manual.out.weight.copy_(model.out.weight[:,keep])
with torch.inference_mode(): manual_out=manual(example)
available=importlib.util.find_spec("torch_pruning") is not None; group_built=False; group_valid=False; group_size=0; version=None; message="torch_pruning not installed"
if available:
    try:
        import torch_pruning as tp
        version=getattr(tp,"__version__","unknown")
        dg=tp.DependencyGraph().build_dependency(model,example_inputs=example)
        group=dg.get_pruning_group(model.a,tp.prune_conv_out_channels,idxs=idxs)
        group_built=True; group_valid=bool(dg.check_pruning_group(group)); group_size=len(group); message=str(group)
    except Exception as exc: message=f"{type(exc).__name__}: {str(exc).splitlines()[0]}"
metrics={"torch_pruning_available":available,"torch_pruning_version":version,"group_built":group_built,"group_valid":group_valid,"group_size":group_size,"manual_output_channels":int(manual_out.shape[1]),"manual_parameters":count_params(manual),"full_parameters":count_params(model),"probe_message":message[:1000]}
analysis=(f"The manual dependency control reduced the model from {metrics['full_parameters']:,} to {metrics['manual_parameters']:,} "
          f"parameters and produced {metrics['manual_output_channels']} output channels. Torch-Pruning availability was {available}; "
          f"group built/valid were {group_built}/{group_valid}. This is a bounded compatibility result when the optional package is absent.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Torch-Pruning available | no |
| Group built | no |
| Group valid | no |
| Manual output channels | 6 |
| Manual parameters | 368 |
| Probe message | torch_pruning not installed |


## 7. Interpret rather than merely print

The manual dependency control reduced the model from 552 to 368 parameters and produced 6 output channels. Torch-Pruning availability was False; group built/valid were False/False. This is a bounded compatibility result when the optional package is absent.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`compatibility-probe`**. The notebook records real package/API availability and preserves the native success or failure state. Missing backend execution remains unmeasured.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 15,
    "title": 'Torch-Pruning DepGraph: A Structured-Pruning Compatibility Lab',
    "environment": ENV,
    "evidence_label": 'compatibility-probe',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'A dependency graph is valuable when its real group, mutation, and save/load path are observed—not when its name appears in a plan.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 15,
  "title": "Torch-Pruning DepGraph: A Structured-Pruning Compatibility Lab",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260823
  },
  "evidence_label": "compatibility-probe",
  "metrics": {
    "torch_pruning_available": false,
    "torch_pruning_version": null,
    "group_built": false,
    "group_valid": false,
    "group_size": 0,
    "manual_output_channels": 6,
    "manual_parameters": 368,
    "full_parameters": 552,
    "probe_message": "torch_pruning not installed"
  },
  "analysis": "The manual dependency control reduced the model from 552 to 368 parameters and produced 6 output channels. Torch-Pruning availability was False; group built/valid were False/False. This is a bounded compatibility result when the optional package is absent.",
  "conclusion": "A dependency graph is valuable when its real group, mutation, and save

## 9. Make the bounded decision

> A dependency graph is valuable when its real group, mutation, and save/load path are observed—not when its name appears in a plan.

**Acceptance/rollback:** Accept the automated route only when the group is valid, forward and quality checks pass, and the mutated architecture has a tested save/load contract.

**Failure analysis:** A successful trace can miss data-dependent control flow or static attributes used outside tensor operations. A package import proves nothing about a specific model group. Conversely, package absence does not falsify the DepGraph method; it only leaves that native path unexecuted.


## 10. Extend the evidence

Install the pinned Torch-Pruning version, run the notebook again, compare printed group operations with the manual ledger, and test whole-model serialization and reload.

The full evidence boundary and references are in [`README.md`](README.md).
